### MTD Performance vs Previous MTD

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_mtd_performance AS
WITH latest_invoice AS (
    SELECT 
        MAX(invoice_ts) AS latest_date,
        DATE_TRUNC('month', MAX(invoice_ts)) AS current_month_start,
        DAY(MAX(invoice_ts)) AS latest_day_of_month
    FROM `03_gold_catalog`.datacube.fact_invoice_dim
),
current_mtd AS (
    SELECT
        fi.store_name,
        fi.manager_name,
        SUM(fi.invoice_amount) AS mtd_revenue,
        COUNT(DISTINCT fi.order_id) AS mtd_completed_orders,
        DATE_TRUNC('month', fi.invoice_ts) AS month_start
    FROM `03_gold_catalog`.datacube.fact_invoice_dim fi
    CROSS JOIN latest_invoice li
    WHERE DATE_TRUNC('month', fi.invoice_ts) = li.current_month_start
      AND DAY(fi.invoice_ts) <= li.latest_day_of_month 
    GROUP BY fi.store_name, fi.manager_name, DATE_TRUNC('month', fi.invoice_ts)
),
previous_mtd AS (
    SELECT
        fi.store_name,
        fi.manager_name,
        SUM(fi.invoice_amount) AS prev_mtd_revenue,
        COUNT(DISTINCT fi.order_id) AS prev_mtd_completed_orders
    FROM `03_gold_catalog`.datacube.fact_invoice_dim fi
    CROSS JOIN latest_invoice li
    WHERE DATE_TRUNC('month', fi.invoice_ts) = ADD_MONTHS(li.current_month_start, -1)
      AND DAY(fi.invoice_ts) <= li.latest_day_of_month 
    GROUP BY fi.store_name, fi.manager_name
)
SELECT
    c.store_name,
    c.manager_name,
    c.mtd_revenue AS current_mtd_revenue,
    c.mtd_completed_orders AS current_mtd_orders,
    COALESCE(p.prev_mtd_revenue, 0) AS previous_mtd_revenue,
    COALESCE(p.prev_mtd_completed_orders, 0) AS previous_mtd_orders,
    c.mtd_revenue - COALESCE(p.prev_mtd_revenue, 0) AS revenue_change,
    ROUND(
        (c.mtd_revenue - COALESCE(p.prev_mtd_revenue, 0)) / NULLIF(p.prev_mtd_revenue, 0),
        2
    ) AS revenue_growth_rate,
    c.month_start AS current_month
FROM current_mtd c
LEFT JOIN previous_mtd p 
    ON c.store_name = p.store_name 
    AND c.manager_name = p.manager_name;

In [0]:
%sql
select * from `03_gold_catalog`.gold_kpi.v_mtd_performance

### Average Days in Shop

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_avg_days_in_shop AS
SELECT
    fo.store_name,
    fo.store_id,
    fo.service_type,
    COUNT(*) AS total_orders,
    ROUND(AVG(fo.days_in_shop), 2) AS avg_days_in_shop
FROM `03_gold_catalog`.datacube.fact_order_dim fo
WHERE fo.order_status = 'completed'  
GROUP BY fo.store_id, fo.store_name, fo.service_type;

In [0]:
%sql
select * from `03_gold_catalog`.gold_kpi.v_avg_days_in_shop

### Survey Coverage

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_survey_coverage AS
SELECT
    fs.store_name,
    COUNT(*) AS surveys_sent,
    SUM(CASE WHEN fs.responded_flag = TRUE THEN 1 ELSE 0 END) AS surveys_responded,
    ROUND(SUM(CASE WHEN fs.responded_flag = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS response_rate
FROM `03_gold_catalog`.datacube.fact_survey_dim fs
GROUP BY fs.store_name;

In [0]:
%sql
SELECT * FROM `03_gold_catalog`.gold_kpi.v_survey_coverage

### Survey Score Summary

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_survey_scores AS
SELECT
    fs.store_name,
    ROUND(AVG(fs.overall_satisfaction_rating), 2) AS avg_satisfaction,
    ROUND(AVG(fs.cleanliness_rating), 2) AS avg_cleanliness,
    ROUND(AVG(fs.communication_rating), 2) AS avg_communication,
    ROUND(AVG(fs.work_quality_rating), 2) AS avg_work_quality,
    ROUND(AVG(fs.delivered_on_time_rating), 2) AS avg_on_time,
    DENSE_RANK() OVER (ORDER BY AVG(fs.overall_satisfaction_rating) DESC) AS satisfaction_rank
FROM `03_gold_catalog`.datacube.fact_survey_dim fs
WHERE fs.responded_flag = TRUE
GROUP BY fs.store_name;

In [0]:
%sql
SELECT * FROM `03_gold_catalog`.gold_kpi.v_survey_scores

### Revenue vs Budget


In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_revenue_vs_budget AS
WITH store_level AS (
    SELECT
        fi.manager_name,
        fi.invoice_year,
        fi.invoice_month,
        fi.store_id,
        MAX(b.budget_amount) AS store_budget,
        SUM(fi.invoice_amount) AS store_revenue
    FROM `03_gold_catalog`.datacube.fact_invoice_dim fi
    JOIN `03_gold_catalog`.gold.dim_budget b 
        ON fi.store_id = b.store_id
        AND fi.invoice_year = b.budget_year
        AND fi.invoice_month = b.budget_month
    GROUP BY fi.manager_name, fi.invoice_year, fi.invoice_month, fi.store_id
)
SELECT
    manager_name,
    invoice_year AS budget_year,
    invoice_month AS budget_month,
    SUM(store_budget) AS budget_amount,
    SUM(store_revenue) AS actual_revenue,
    ROUND(SUM(store_revenue) / SUM(store_budget), 2) AS achievement_ratio,
    DENSE_RANK() OVER (
        ORDER BY SUM(store_revenue) / SUM(store_budget) DESC
    ) AS manager_rank
FROM store_level
GROUP BY manager_name, invoice_year, invoice_month;

In [0]:
%sql
SELECT * FROM `03_gold_catalog`.gold_kpi.v_revenue_vs_budget

### Top Technicians by Completion Time Accuracy

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_top_technicians_accuracy AS
SELECT
    fo.store_name,
    fo.technician_name,
    AVG(fo.planned_vs_actual_completion_days) AS avg_completion_variance,
    DENSE_RANK() OVER (
        PARTITION BY fo.store_name ORDER BY ABS(AVG(fo.planned_vs_actual_completion_days))
    ) AS accuracy_rank
FROM `03_gold_catalog`.datacube.fact_order_dim fo
GROUP BY fo.store_name, fo.technician_name
QUALIFY accuracy_rank <= 5;

In [0]:
%sql
SELECT * FROM `03_gold_catalog`.gold_kpi.v_top_technicians_accuracy

### Year‑to‑Date Revenue Growth 

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_ytd_revenue_growth AS
SELECT
    fi.store_name,
    SUM(CASE WHEN fi.invoice_year = YEAR(CURRENT_DATE()) THEN fi.invoice_amount END)
        AS current_ytd_revenue,
    SUM(CASE WHEN fi.invoice_year = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END)
        AS last_ytd_revenue,
    ROUND(
        (
            SUM(CASE WHEN fi.invoice_year = YEAR(CURRENT_DATE()) THEN fi.invoice_amount END) -
            SUM(CASE WHEN fi.invoice_year = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END)
        ) / 
        SUM(CASE WHEN fi.invoice_year = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END),
        2
    ) AS ytd_growth_rate,
    DENSE_RANK() OVER (
        ORDER BY (
            SUM(CASE WHEN fi.invoice_year = YEAR(CURRENT_DATE()) THEN fi.invoice_amount END) -
            SUM(CASE WHEN fi.invoice_year = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END)
        ) DESC
    ) AS growth_rank
FROM `03_gold_catalog`.datacube.fact_invoice_dim fi
GROUP BY fi.store_name;

In [0]:
%sql
SELECT * FROM  `03_gold_catalog`.gold_kpi.v_ytd_revenue_growth

### Stage‑wise Cycle Time

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_stage_cycle_time AS
SELECT
    fo.store_name,
    fo.service_type,
    ROUND(AVG(fo.vehicle_in_to_work_start_days), 2) AS avg_vehicle_in_to_start,
    ROUND(AVG(fo.work_start_to_completion_days), 2) AS avg_work_start_to_completion,
    ROUND(AVG(fo.work_completion_to_delivery_days), 2) AS avg_completion_to_delivery
FROM `03_gold_catalog`.datacube.fact_order_dim fo
WHERE fo.order_status = 'completed'  
GROUP BY fo.store_name, fo.service_type;

In [0]:
%sql
SELECT * FROM `03_gold_catalog`.gold_kpi.v_stage_cycle_time

###  Estimator Accuracy

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_estimator_accuracy AS
SELECT
    fo.estimator_name,
    COUNT(*) AS total_estimates,
    ROUND(AVG(fo.final_estimate_amount - fo.initial_estimate_amount), 2) AS avg_estimate_variance,
    ROUND(AVG((fo.final_estimate_amount - fo.initial_estimate_amount) / NULLIF(fo.initial_estimate_amount, 0)), 2) AS avg_variance_pct,
    DENSE_RANK() OVER (ORDER BY ABS(AVG((fo.final_estimate_amount - fo.initial_estimate_amount) / NULLIF(fo.initial_estimate_amount, 0)))) AS estimator_rank
FROM `03_gold_catalog`.datacube.fact_order_dim fo
WHERE fo.initial_estimate_amount IS NOT NULL 
  AND fo.final_estimate_amount IS NOT NULL
GROUP BY fo.estimator_name;

In [0]:
%sql
SELECT * FROM `03_gold_catalog`.gold_kpi.v_estimator_accuracy

### Technician Workload

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_technician_workload AS
SELECT
    fo.store_name,
    fo.technician_name,
    COUNT(*) AS orders_handled,
    SUM(fo.days_in_shop) AS total_days_in_shop,
    DENSE_RANK() OVER (PARTITION BY fo.store_name ORDER BY COUNT(*) DESC) AS workload_rank
FROM `03_gold_catalog`.datacube.fact_order_dim fo
GROUP BY fo.store_name, fo.technician_name;

In [0]:
%sql
SELECT * FROM `03_gold_catalog`.gold_kpi.v_technician_workload